In [ ]:
You're working at a retail chain analyzing inventory turnover. The store manager needs to understand stock movement patterns to optimize ordering. 

Data Structure: DataFrame

product_id
category
stock_level
last_restock_date
sales_last_30_days
supplier_lead_time
unit_cost
Task: Generate code to:

Calculate inventory turnover rates.
Identify slow-moving items.
Predict potential stockouts.
Create visualizations.                                                                                                                                          
Include error handling, proper date calculations and explain the visualizations choosen

In [ ]:
Here’s a **step-by-step PySpark + visualization solution** for your retail inventory analysis. It includes:

✅ Inventory turnover calculation
✅ Slow-moving item detection
✅ Potential stockout prediction
✅ Visualizations
✅ Error handling
✅ Proper date calculations
✅ Explanation of charts

```python
# ================================
# RETAIL INVENTORY ANALYSIS
# ================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# Error Handling: Start Spark
# -------------------------------
try:
    spark = SparkSession.builder \
        .appName("Retail Inventory Analysis") \
        .getOrCreate()

    print("Spark Session Started Successfully")

except Exception as e:
    print(f"Error starting Spark Session: {e}")


# -------------------------------
# Sample Data
# -------------------------------
data = [
    (101, "Electronics", 50, "2026-05-01", 120, 10, 1500),
    (102, "Clothing", 200, "2026-04-25", 20, 7, 40),
    (103, "Groceries", 30, "2026-05-10", 300, 3, 5),
    (104, "Furniture", 80, "2026-03-15", 5, 14, 2500),
    (105, "Beauty", 60, "2026-04-20", 90, 5, 25)
]

columns = [
    "product_id",
    "category",
    "stock_level",
    "last_restock_date",
    "sales_last_30_days",
    "supplier_lead_time",
    "unit_cost"
]

try:
    df = spark.createDataFrame(data, columns)

except Exception as e:
    print(f"Error creating DataFrame: {e}")


# -------------------------------
# Date Calculations
# -------------------------------
try:
    df = df.withColumn(
        "last_restock_date",
        to_date(col("last_restock_date"), "yyyy-MM-dd")
    )

    df = df.withColumn(
        "days_since_restock",
        datediff(current_date(), col("last_restock_date"))
    )

except Exception as e:
    print(f"Date conversion error: {e}")


# -------------------------------
# 1. Inventory Turnover Rate
# Formula:
# Sales Last 30 Days / Stock Level
# -------------------------------
try:
    df = df.withColumn(
        "inventory_turnover_rate",
        when(col("stock_level") > 0,
             col("sales_last_30_days") / col("stock_level"))
        .otherwise(0)
    )

except Exception as e:
    print(f"Inventory turnover calculation error: {e}")


# -------------------------------
# 2. Identify Slow-Moving Items
# Rule:
# Turnover rate < 0.5
# -------------------------------
try:
    slow_moving = df.filter(
        col("inventory_turnover_rate") < 0.5
    )

    print("Slow Moving Items")
    slow_moving.show()

except Exception as e:
    print(f"Slow-moving analysis error: {e}")


# -------------------------------
# 3. Predict Potential Stockouts
# Rule:
# Estimated days remaining
# < supplier lead time
# -------------------------------
try:
    df = df.withColumn(
        "daily_sales_rate",
        col("sales_last_30_days") / 30
    )

    df = df.withColumn(
        "days_until_stockout",
        when(col("daily_sales_rate") > 0,
             col("stock_level") / col("daily_sales_rate"))
        .otherwise(None)
    )

    stockout_risk = df.filter(
        col("days_until_stockout") < col("supplier_lead_time")
    )

    print("Potential Stockout Risk")
    stockout_risk.show()

except Exception as e:
    print(f"Stockout prediction error: {e}")


# -------------------------------
# Convert to Pandas for Visualization
# -------------------------------
try:
    pdf = df.toPandas()

except Exception as e:
    print(f"Conversion to Pandas failed: {e}")


# -------------------------------
# 4. Visualizations
# -------------------------------

# Inventory Turnover Rate
plt.figure(figsize=(8,5))
plt.bar(
    pdf["category"],
    pdf["inventory_turnover_rate"]
)
plt.title("Inventory Turnover Rate by Category")
plt.xlabel("Category")
plt.ylabel("Turnover Rate")
plt.xticks(rotation=45)
plt.show()


# Slow-Moving Items
slow_pdf = slow_moving.toPandas()

plt.figure(figsize=(8,5))
plt.bar(
    slow_pdf["category"],
    slow_pdf["inventory_turnover_rate"]
)
plt.title("Slow-Moving Inventory")
plt.xlabel("Category")
plt.ylabel("Turnover Rate")
plt.xticks(rotation=45)
plt.show()


# Stockout Prediction
plt.figure(figsize=(8,5))
plt.scatter(
    pdf["supplier_lead_time"],
    pdf["days_until_stockout"]
)
plt.title("Stockout Risk Analysis")
plt.xlabel("Supplier Lead Time")
plt.ylabel("Days Until Stockout")
plt.show()
```

## Explanation of the Visualizations

### 1. Bar Chart: Inventory Turnover Rate

**Why chosen?**
A **bar chart** is ideal for comparing turnover rates across categories.

**What it shows:**

* High bars = products selling fast
* Low bars = products moving slowly

Example:

* Groceries may have a high turnover.
* Furniture may have a low turnover.

---

### 2. Bar Chart: Slow-Moving Inventory

**Why chosen?**
Helps managers quickly spot products that are not selling well.

**What it shows:**

* Categories/products with turnover below the threshold (`< 0.5`)
* Helps reduce overstocking and storage costs.

---

### 3. Scatter Plot: Stockout Risk

**Why chosen?**
A **scatter plot** helps compare two variables:

* Supplier lead time
* Days until stockout

**What it shows:**

* Points below the “safe zone” indicate products likely to run out before new stock arrives.
* Managers can prioritize urgent reordering.

### Business Insight

This analysis helps the retail manager:

1. **Optimize ordering** by predicting shortages.
2. **Reduce dead stock** by identifying slow-moving items.
3. **Improve cash flow** through better inventory turnover monitoring.
